In [8]:
"""Kiểm tra xem đã cho vectorstore cho rxnorm đúng với mong muốn hay chưa"""
import sys
from pathlib import Path
import sqlite3
import json

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT / "src"))

from Database.RxNorm.build_chroma import fetch_rxnorm_objects, object_to_document

DB_PATH = PROJECT_ROOT / "data" / "db" / "RxNorm" / "rxnorm.db"


conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

In [12]:
"""Kiểm tra xem đã tạo rxnorm object đúng trường hay chưa"""
try:
    with sqlite3.connect(DB_PATH) as conn:
        query = """
            SELECT 
                count(*)
            FROM rxnconso
            WHERE LAT = 'ENG'
            AND SUPPRESS = 'N'
            AND STR IS NOT NULL
        """
        cursor.execute(query)
        row_count = cursor.fetchone()[0]
        print(f"Total rows = {row_count:,}")
except sqlite3.Error as e:
    print(e)

objects = fetch_rxnorm_objects(DB_PATH)
print(f"Total objects = {len(objects):,}")
print(json.dumps(objects[:1], indent=2, ensure_ascii=False))
print("CHUYỂN OBJECT ĐÚNG")

Total rows = 806,742
Total objects = 806,742
[
  {
    "RXCUI": "3",
    "LAT": "ENG",
    "TS": "",
    "LUI": "",
    "STT": "",
    "SUI": "",
    "ISPREF": "",
    "RXAUI": "8717795",
    "SAUI": "",
    "SCUI": "58488005",
    "SDUI": "",
    "SAB": "SNOMEDCT_US",
    "TTY": "PT",
    "CODE": "58488005",
    "STR": "1,4-alpha-Glucan branching enzyme",
    "SRL": "",
    "SUPPRESS": "N",
    "CVF": ""
  }
]


In [15]:
"""
Kiểm tra định dạng của langchain document có được
định dạng đúng trước khi chuyển thành vector store chưa
"""
test_object = objects[0]
doc = object_to_document(test_object)

print(doc.page_content) # Đây là thông tin sẽ được nhét vào vector store
print(json.dumps(doc.metadata, indent=2, ensure_ascii=False))

Drug name: 1,4-alpha-Glucan branching enzyme
        RxCUI: 3
        Term type: PT
        Source: SNOMEDCT_US
        Code: 58488005
{
  "rxcui": "3",
  "tty": "PT",
  "sab": "SNOMEDCT_US",
  "code": "58488005",
  "drug_name": "1,4-alpha-Glucan branching enzyme"
}
